# Practice Pipeline: Avibacterium paragallinarum

## Reference Paper
Hashish et al. (2023). Complete genome sequences generated using hybrid
Nanopore-Illumina assembly of two non-typical Avibacterium paragallinarum
strains isolated from clinically normal chicken flocks.
Microbiology Resource Announcements, 12(10): e00128-23.
DOI: 10.1128/MRA.00128-23

## Organism
**Avibacterium paragallinarum** is a Gram-negative bacterium belonging to
the family Pasteurellaceae. It is the causative agent of infectious coryza
(IC), an acute respiratory disease in chickens. This strain (npAP/GA-USA/
20231216/S1-1) was isolated from a naive, healthy layer chicken in Georgia,
USA in 2023 — showing no clinical signs of disease.

## BioProject
PRJNA1177890
https://www.ncbi.nlm.nih.gov/bioproject/PRJNA1177890

## Sequencing Data
| Platform | SRR | Size | Link |
|---|---|---|---|
| Illumina MiSeq (Paired-End) | SRR31123925 | 710.6 MB | https://www.ncbi.nlm.nih.gov/sra/SRX26505951 |
| Oxford Nanopore MinION (Single-End) | SRR31139160 | 1,019.9 MB | https://trace.ncbi.nlm.nih.gov/Traces/?run=SRR31139160 |

## Pipeline Tools (same as Gluconobacter cerinus paper)
- Quality Control: FastQC (Illumina), NanoPlot (Nanopore), MultiQC (combined)
- Trimming: Trimmomatic (Illumina), Porechop (Nanopore)
- Filtering: Filtlong (Nanopore)

In [ ]:
# Installing the SRA Toolkit
!sudo apt-get install sra-toolkit -y

In [ ]:
print("Downloading subsamples...\n")

# Illumina (Paired-End) - first 50,000 reads:
!fastq-dump --split-files -X 50000 SRR31123925

# Nanopore (Single-End) - first 50,000 reads:
!fastq-dump -X 50000 SRR31139160

print("\nDone! Refresh the left panel and check the file names.")

## Step 2: Subsampling (Downsampling) — Downloading 50,000 Reads

### What did we just do?
Using the SRA Toolkit's `fastq-dump` command, we downloaded only the first
50,000 reads from each sequencing run directly from NCBI's servers — without
downloading the full dataset.

### What files do we have now?
- **SRR31123925_1.fastq** → Illumina MiSeq Forward reads (R1)
- **SRR31123925_2.fastq** → Illumina MiSeq Reverse reads (R2)
- **SRR31139160.fastq**  → Oxford Nanopore MinION reads

### Why did we subsample?
The full datasets are large (Illumina: ~710 MB, Nanopore: ~1 GB).
Downloading and processing the entire data would take a long time and
consume significant memory. Instead, we extracted a small representative
subset (50,000 reads) to quickly assess the overall quality of the data.
This is statistically sufficient because sequencing reads are randomly
distributed across the flow cell — systematic errors and adapter
contamination will appear in the first 50,000 reads just as clearly
as in the full dataset.

### What are we going to do next?
We will run quality control (QC) on these subsampled reads:
- **FastQC** → to assess the quality of the Illumina short reads
- **NanoPlot** → to assess the quality of the Nanopore long reads
- **MultiQC** → to combine all QC reports into a single interactive HTML report

These three reports together will form **Set 1** of our analysis.

In [ ]:
# Install FastQC for short-read quality control
!sudo apt-get update
!sudo apt-get install fastqc -y

In [ ]:
# Create output folder and run FastQC on both Illumina files
!mkdir -p qc_reports

print("Running FastQC on Illumina reads...\n")
!fastqc SRR31123925_1.fastq SRR31123925_2.fastq -o qc_reports/

print("\nFastQC completed! Check qc_reports/ folder for HTML reports.")

In [ ]:
# Install NanoPlot for long-read quality control
print("Installing NanoPlot...\n")
!pip install NanoPlot

print("\nRunning NanoPlot on Nanopore reads...\n")
!NanoPlot --fastq SRR31139160.fastq -o qc_reports/nanoplot_result --threads 2

print("\nNanoPlot completed!")

In [ ]:
# Install MultiQC to combine all QC reports
print("Installing MultiQC...\n")
!pip install multiqc
print("\nMultiQC installed!")

## Step 3: Quality Control Results — What Did We Find?

### What we produced (Set 1 — Subsampled Data):
- **SRR31123925_1_fastqc.html** → FastQC report for Illumina Forward reads (R1)
- **SRR31123925_2_fastqc.html** → FastQC report for Illumina Reverse reads (R2)
- **NanoPlot-report.html** → NanoPlot report for Nanopore reads
- **multiqc_report.html** → Combined interactive summary of all reports

---

### What do these reports tell us?
**FastQC (Illumina):**
- Per-base sequence quality → Are the bases reliable across read length?
- Adapter content → Did any Illumina adapter sequences contaminate the reads?
- GC content → Does the GC distribution match what we expect for this organism?
- R2 typically shows lower quality toward the end — this is normal for
  paired-end sequencing due to flow cell degradation over time.

**NanoPlot (Nanopore):**
- Read length distribution → How long are the Nanopore reads on average?
- N50 value → Half of all sequenced bases are in reads longer than this value.
- Quality score distribution → What is the overall accuracy of the long reads?

**MultiQC:**
- Aggregates all reports into one dashboard for easy comparison.

---

### What can you do with this data next?

**1. Trimming and Filtering (Set 3 — Post-QC):**
Apply the same tools used in the Gluconobacter cerinus paper:
- Trimmomatic → remove adapters and low-quality bases from Illumina reads
- Porechop → remove Nanopore adapters
- Filtlong → filter out short and low-quality Nanopore reads
Then re-run FastQC + NanoPlot + MultiQC to compare before vs. after cleaning.

**2. Genome Assembly:**
After cleaning, you can assemble the complete genome using:
- Unicycler → hybrid assembler that combines Illumina + Nanopore reads
- The result will be a complete circular chromosome of A. paragallinarum

**3. Genome Annotation:**
Annotate the assembled genome using:
- NCBI PGAP → identifies coding sequences, tRNAs, rRNAs
- Prokka → faster local alternative for bacterial genome annotation

---

### How can you modify the code for other organisms?

| What to change | How |
|---|---|
| Different organism | Replace SRR31123925 and SRR31139160 with new SRR numbers from NCBI SRA |
| More reads | Change `-X 50000` to `-X 100000` for a larger subsample |
| Different output folder | Change `qc_reports/` to any folder name you prefer |
| Skip NanoPlot | If you only have Illumina data, remove the NanoPlot step entirely |
| Multiple samples | Add more `fastq-dump` lines with different SRR numbers and run FastQC on all at once — MultiQC will combine everything automatically |

---

### Key takeaway
This pipeline is fully reusable. Any bacterial genome project deposited
in NCBI SRA with both Illumina and Nanopore data can be analyzed with
exactly these same steps — just swap the SRR accession numbers.

# Step 4: Trimming & Filtering — Why and How

## Why do we trim?

Raw sequencing reads come directly off the machine and are never perfect.
They contain:
- **Adapter sequences** — synthetic DNA added during library prep, not from your organism
- **Low-quality bases** — especially near the 3' end of reads
- **Very short reads** — too short to map to the genome reliably

Trimming removes all of this before any downstream analysis.

---

## What happens if you skip trimming?

Without trimming, your aligner or assembler sees contamination as real biological sequence. Common consequences:
- Adapter sequences cause reads to **fail alignment**, lowering your mapping rate
- Low-quality bases introduce **false SNPs and indels** in variant calling
- Assembly software may produce **chimeric or fragmented contigs**
- Simply put: garbage in, garbage out.

---

## What does trimming actually do?

It scans each read and:
- Detects and removes adapter contamination from the 3' end
- Cuts bases whose quality score falls below a threshold (e.g. Q20 = 99% base accuracy)
- Discards reads shorter than a minimum length (e.g. 30 bp)
- For paired-end data: discards **both** reads of a pair if either fails

---

## How do we choose the right tool?

The choice depends on **read type**, not organism:

| Read Type | Platform | Tool Used |
|-----------|----------|-----------|
| Short read (50–300 bp) | Illumina | **Cutadapt** |
| Long read (1–50 kb) | Nanopore | **NanoFilt** |

Our dataset has both — so we run both tools, one after the other.

---

## What if this were human genome instead of *Avibacterium paragallinarum*?

The trimming step would be **identical** — same Cutadapt and NanoFilt commands, same parameters.
The difference comes **after** trimming:

- **Reference size:** human genome is ~3 Gb vs ~2.3 Mb for this bacterium — alignment takes far longer
- **Repeats:** ~50% of human DNA is repetitive; reads from repeat regions are harder to place uniquely
- **Ploidy:** humans are diploid (two copies of each chromosome); bacteria are haploid
- **Coverage:** 30× is standard for human WGS; a small bacterial genome can be covered much more deeply with the same data

> Trimming logic is biology-agnostic. The complexity comes later, in alignment and variant calling.

In [ ]:
# Installing Cutadapt
print("Installing the short-read noise removal tool (Cutadapt)...\n")

!sudo apt-get update
!sudo apt-get install cutadapt -y

print("\nGreat! Cutadapt installation completed successfully.")

In [ ]:
# Illumina Trimming with Cutadapt
!mkdir -p filtered_reads

print("TRIMMING AND FILTERING the ILLUMINA (Paired-End) raw data...\n")

!cutadapt -a AGATCGGAAGAG -A AGATCGGAAGAG \
    -q 20 -m 30 \
    -o filtered_reads/filtered_SRR31123925_1.fastq \
    -p filtered_reads/filtered_SRR31123925_2.fastq \
    SRR31123925_1.fastq SRR31123925_2.fastq

print("\nIllumina trimming complete. Outputs are in the 'filtered_reads/' folder.")

In [ ]:
# Nanopore Long Read Filtering - NanoFilt
!pip install nanofilt

print("Filtering NANOPORE reads...\n")

!NanoFilt -q 8 -l 500 SRR31139160.fastq > filtered_reads/filtered_SRR31139160.fastq

print("\nNanopore filtering complete. Output: filtered_reads/filtered_SRR31139160.fastq")

# Re-QC — Did Trimming Actually Work?

## Why do we run QC again?

After trimming, we don't just assume the data is clean — we **verify** it.
Running FastQC and NanoPlot on the filtered reads lets us compare before and after,
and confirm that trimming did what we expected.

---

## What are we checking?

We run the exact same tools as before (FastQC → NanoPlot → MultiQC), but this time
on the **filtered_reads/** files instead of the raw ones.

If trimming worked correctly, we expect to see:

- ✅ **No adapter contamination** — the adapter content graph should be flat
- ✅ **Higher average quality scores** — especially near read ends
- ✅ **Slightly fewer reads** — the ones that were too short or too low quality are gone
- ✅ **More uniform read length distribution** — very short reads removed

---

## What if the quality looks the same?

That can happen — and it's not always a problem.
It may mean the raw data was already quite clean, or that only a small percentage
of reads were affected. What matters is that adapter content is gone.

---

## Why MultiQC at the end?

MultiQC combines all individual FastQC and NanoPlot reports into a single HTML file.
This makes it easy to compare Illumina and Nanopore results side by side,
and to see the before/after difference at a glance.

> Re-QC is not optional. It is the checkpoint that confirms your pipeline
> is producing trustworthy input for assembly or alignment.

In [ ]:
# Re-QC: Illumina filtered reads
print("Running FastQC on the filtered Illumina reads...\n")

!fastqc filtered_reads/filtered_SRR31123925_1.fastq \
         filtered_reads/filtered_SRR31123925_2.fastq \
         -o qc_reports/

print("\nFastQC completed!")

In [ ]:
# MultiQC - combine all reports
print("Building the MultiQC report...\n")

!multiqc qc_reports/ -o qc_reports/multiqc_filtered

print("\nMultiQC completed! Check the qc_reports/multiqc_filtered/ folder.")

# De Novo Genome Assembly — Why We Assemble Instead of Map

## What is genome assembly?

Assembly is the process of taking millions of short, overlapping DNA reads
and computationally piecing them together into longer, continuous sequences
called **contigs** — and ideally into a complete chromosome.

Think of it like solving a jigsaw puzzle:
- Each read is one puzzle piece
- The assembler finds overlapping edges between pieces
- The final picture is the reconstructed genome

---

## Why are we assembling instead of mapping?

In the previous step, we trimmed and filtered our reads. Now we have two choices:

| Approach | When to use |
|----------|-------------|
| **Reference-based mapping** | A high-quality reference genome already exists for your exact strain |
| **De novo assembly** | You are sequencing a novel strain with no existing complete reference |

Our BioProject (PRJNA1177890) sequenced **non-pathogenic *Avibacterium paragallinarum* strains
isolated from healthy chickens in the USA** — strains that had never been fully sequenced before.
There is no complete, published genome for these exact isolates.
Therefore, we cannot map to something that does not exist yet.
We must build the genome from scratch. That is de novo assembly.

---

## Why do we have both Illumina and Nanopore reads?

This is called a **hybrid assembly** strategy — and it is one of the most powerful
approaches in modern microbial genomics.

Each platform has complementary strengths:

| Platform | Read Length | Accuracy | Strength |
|----------|-------------|----------|----------|
| Illumina (short read) | ~150 bp | Very high (Q30+) | Corrects sequencing errors |
| Oxford Nanopore (long read) | 10,000–50,000 bp | Lower (Q10–Q15) | Spans repetitive regions, resolves genome structure |

Short reads alone cannot bridge repetitive DNA regions — the assembler gets confused
and produces many fragmented contigs.
Long reads alone contain too many errors to assemble accurately.
Together, long reads provide the structural backbone and short reads polish the errors.
The result is a **complete, accurate genome**.

---

## What tool will we use?

We will use **Unicycler** — the gold standard for hybrid bacterial genome assembly.

Unicycler was specifically designed for small bacterial genomes and handles
Illumina + Nanopore data natively. Under the hood it:
1. Builds an assembly graph from Illumina short reads (via SPAdes)
2. Uses Nanopore long reads to resolve ambiguous connections in the graph
3. Circularizes the chromosome and any plasmids it finds
4. Outputs a clean FASTA file of the final assembly

---

## What if this were a human genome?

De novo assembly of a human genome is orders of magnitude harder:
- The human genome is ~3 Gb vs ~2 Mb for this bacterium (1,500x larger)
- ~50% of it is repetitive sequence — extremely difficult to resolve
- It is diploid — two copies of every chromosome must be disentangled
- Tools like **Hifiasm** or **verkko** are used instead of Unicycler
- Assembly requires weeks of compute time and hundreds of GB of RAM

For bacteria, Unicycler can produce a complete genome in under an hour on Colab.
This is one reason microbial genomics has advanced so rapidly.

> De novo assembly turns raw sequencing data into a new entry in the tree of life.
> Every complete genome you produce is a permanent scientific contribution.

## Step 5: Trimming & Filtering — Decision Making and Tool Selection

### Why are we doing this step?
Raw sequencing data is never perfect. Before we can assemble a genome or
map reads to a reference, we must remove technical noise introduced during
library preparation and sequencing. This step is called pre-processing or
cleaning, and it is non-negotiable in any serious bioinformatics pipeline.

Without this step:
- Adapter sequences would create false overlaps in genome assembly
- Low-quality bases would introduce errors into our final results
- Very short reads would add noise without contributing useful information

---

### What am I actually learning by doing this?

**1. Critical thinking about tool selection:**
Not every paper uses the same tool. You are learning to read a Methods
section and identify exactly which tool was used and why — then reproduce
it yourself.

**2. Understanding parameter logic:**
Every number in a trimming command has a biological meaning. You are not
just copying commands — you are learning what SLIDINGWINDOW:4:20 means
(scan 4 bases at a time, cut when average quality drops below Phred 20)
and why MINLEN:30 matters (reads shorter than 30 bp are useless for assembly).

**3. Paired-end awareness:**
Illumina produces two files (R1 forward + R2 reverse). These must be
processed together to maintain synchronization. If one read is discarded,
its pair must also be discarded or moved to an unpaired file. This is why
Trimmomatic produces 4 output files, not 2.

**4. Platform-specific logic:**
Illumina and Nanopore have completely different error profiles and adapter
types. You are learning that the same pipeline step requires completely
different tools depending on which sequencing machine was used.

---

### How do I decide which tool to use?

**Step 1: Read the Methods section of your paper.**
The paper always tells you which tool was used. This is your primary source.

**Step 2: Identify the sequencing kit.**
The kit determines which adapter sequences contaminate your data.
Different kits → different adapter files → different parameters.

**Step 3: Match the tool to the platform.**

| Platform | Adapter Removal | Quality Filtering |
|---|---|---|
| Illumina | Trimmomatic or Cutadapt | Built into the same tool |
| Nanopore | Porechop | Filtlong (separate step) |
| PacBio HiFi | Not needed | Filtlong if necessary |

**Step 4: Check which adapter file to use.**
Look at the library preparation kit in the paper:

| Kit | Adapter File |
|---|---|
| NEBNext Ultra II (Gluconobacter paper) | TruSeq3-PE.fa |
| Nextera XT (our Avibacterium paper) | NexteraPE-PE.fa |
| TruSeq (classic Illumina) | TruSeq2-PE.fa |

---

### What does our Avibacterium paper say?

Hashish et al. (2023) used:
- **Illumina MiSeq** with the **Nextera XT DNA Library Prep Kit**
  → This means adapters in our data are Nextera adapters
  → We must use NexteraPE-PE.fa in Trimmomatic

- **Oxford Nanopore MinION** with the **Ligation Sequencing Kit (SQK-LSK109)**
  → Standard MinION adapters → Porechop handles these perfectly

- The paper does not specify trimming parameters explicitly, so we follow
  the same logic as the Gluconobacter paper since both use the same
  hybrid assembly approach.

---

### Key differences between our pipeline and the instructor's pipeline:

| Feature | Instructor (Gluconobacter) | Our practice (Avibacterium) |
|---|---|---|
| Illumina kit | NEBNext Ultra II | Nextera XT |
| Adapter file | TruSeq3-PE.fa | NexteraPE-PE.fa |
| Trimming tool | Trimmomatic | Trimmomatic (same) |
| Nanopore tool | Porechop + Filtlong | Porechop + Filtlong (same) |
| Filtlong min_length | 2,000 bp | 2,000 bp (same — both are bacteria) |
| Filtlong keep_percent | 90% | 90% (same strategy) |
| Filtlong target_bases | 1,500,000,000 (1.5 Gb) | ~1,500,000,000 (similar genome size) |
| Reference for Filtlong | Trimmomatic output (Illumina) | Trimmomatic output (Illumina) |

**The only real difference is the adapter file.**
Everything else follows the same logic because both organisms are bacteria
with similar genome sizes (~2-3 Mb) sequenced on the same platforms
(Illumina + MinION R9.4.1).

---

### Bottom line:
Trimming and filtering is not about blindly running commands.
It is about understanding your data's origin, identifying the noise
sources specific to your experiment, and selecting the right tool
with the right parameters to remove only the noise — while preserving
every useful biological signal.

In [ ]:
print("Downloading the FULL raw data...\n")

# Short-read (Illumina, paired-end):
!wget -P full_data "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR31123925/SRR31123925"

# Long-read (Nanopore, single-end):
!wget -P full_data "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR31139160/SRR31139160"

# Don't stop the download until it finishes!
#    Illumina: ~710 MB | Nanopore: ~1 GB
#    Keep the browser tab open.

In [ ]:
# Check the files:
!ls -lh full_data/
!file full_data/*